In [6]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"


In [9]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="google/gemma-4-12b-qat",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.8,
)

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)


In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)


In [11]:
from pprint import pprint

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='781b4502-6f6e-4878-b92e-0d9ba6014bd4'),
              AIMessage(content="To help you with that, could you please provide the email address you'd like me to read?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1227, 'prompt_tokens': 115, 'total_tokens': 1342, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1202, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'google/gemma-4-12b-qat', 'system_fingerprint': 'google/gemma-4-12b-qat', 'id': 'chatcmpl-32624wdatwlwb1ydbrf68', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ed8

In [12]:
print(response['__interrupt__'])

KeyError: '__interrupt__'

In [13]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Approve

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

In [ ]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Edit

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)